# 04 — Explanation and advice · Crowdfund Compass (Group 11)
Goal: turn the final model into things a coach or founder can use: why a campaign scores as it does, six area scores, the single changes that are associated with better odds, and a verdict.

Workflow: 1) Setup → 2) Load model and data → 3) SHAP → 4) Area scores → 5) Chance range and verdict rule → 6) What to change → 7) Product types that do badly → 8) Explanation tree → 9) Examples → 10) Conclusions.

**Read this first:** everything here describes associations in past campaigns. A suggested change is "campaigns that did this were funded more often", never a promise.

Run notebook 03 first (it saves the model). *AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup

In [ ]:
import sys, time, json, warnings, os
warnings.filterwarnings('ignore')
!pip -q install vaderSentiment
import numpy as np, pandas as pd, joblib, xgboost as xgb
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200); pd.set_option('display.max_colwidth', 90)

# from google.colab import drive; drive.mount('/content/drive')
project = 'drive/My Drive/crowdfund_compass'      # folder with features.py, modeling.py, advice.py
out = project + '/models'                         # written by notebook 03
sys.path.insert(0, project)
import features as F, modeling as M, advice as A

## 2) Load the model and the data
Training campaigns (2023–24) are the reference population; the 2025 test year is used to check everything.

In [ ]:
ART=joblib.load(out+'/models_M.joblib'); FM=ART['features']
D=M.prepare(pd.read_parquet(out+'/features.parquet'))
TR=D[D.period.isin(['2023','2024'])&D.has_video.notna()].copy(); TE=D[D.period=='2025'].copy()
oof,(ts,),_=M.text_score(TR,[TE]); TR['text_score']=oof; TE['text_score']=ts
adv=A.Advisor(ART)

## 3) SHAP: what drives the score? (beyond class)
*Why:* a coach needs reasons, not only a number. *In plain words:* SHAP splits each campaign's score into one piece per feature: how much that feature pushed the score up or down compared with an average campaign. The pieces add up exactly to the model's score (checked below).

In [ ]:
# ---- 1) SHAP: global importance
S=adv.shap(TE); imp=S.abs().mean().sort_values(ascending=False)
print('mean |SHAP| top 12 (log-odds):'); print(imp.head(12).round(3).to_string())
G=pd.Series({a:S[c].abs().sum(axis=1).mean() for a,c in A.AREAS.items()}); print('by area, share of total:'); print((G/G.sum()).round(3).sort_values(ascending=False).to_string())
(G/G.sum()).to_csv(out+'/shap_by_area.csv'); imp.to_csv(out+'/shap_by_feature.csv')
# additivity check: SHAP sums + base = model margin
c=ART['model'].get_booster().predict(xgb.DMatrix(TE[FM]),pred_contribs=True); raw=ART['model'].predict_proba(TE[FM])[:,1]
print('SHAP additivity: max |sum - logit(p)| =',float(np.abs(c.sum(axis=1)-np.log(raw/(1-raw))).max().round(5)))

**Interpretation:** what the product is (Market) and who launches it (Founder) account for more than half of the explanation. The areas a founder can change before launch (Ambition, Presentation, Timing, Pitch) account for the rest.

## 4) Area scores, 0–100
For each of the six areas we add up the SHAP pieces of its features and ask: what share of the reference campaigns had a lower sum? 50 = typical, 100 = this area helps more than in every reference campaign. Bands: 0–25 weak (red), 26–60 to improve (amber), 61–100 strong (green). The check below shows that campaigns with a weak area were funded less often in 2025.

In [ ]:
# ---- 2) area scores: reference = training campaigns 2023-24
ref=adv.area_sums(TR); ART['area_reference']={k:np.sort(ref[k].values) for k in A.AREAS}; adv.ref=ART['area_reference']
AS=adv.area_scores(TE); TE=TE.join(AS.add_prefix('score_'))
for a in A.AREAS:
    b=pd.cut(TE['score_'+a],[-1,25,60,100],labels=['weak (0-25)','to improve (26-60)','strong (61-100)'])
    t=TE.groupby(b,observed=True).success.agg(n='size',funded='mean'); print(a, ' | '.join(f'{i}: n={int(r.n)}, funded {r.funded:.2f}' for i,r in t.iterrows()))

## 5) Chance range and the verdict rule
- **High end** of the range: calibrated chance in our sample. **Low end:** the same after the shift to Kickstarter's official rates.
- The ranking of campaigns is solid, the levels are rough. So **Coach first** is decided by rank: the riskiest share of the cohort that the coach has capacity for (default 30%), ranked by the low end.
- To see what that means outside our sample, the 2025 test year is re-weighted so that each parent category has Kickstarter's official share of failures.

In [ ]:
# ---- 3) chances and the verdict numbers
lo,hi=adv.chances(TE); TE['low']=lo; TE['high']=hi
s25=TE.groupby('parent_category').success.mean()
TE['w']=np.where(TE.success==0,TE.parent_category.map(lambda c:(s25[c]/(1-s25[c]))/(M.OFFICIAL_RATE[c]/(1-M.OFFICIAL_RATE[c]))),1.)   # re-weight to Kickstarter's official mix
print('re-weighted cohort: share failing',round((TE.w*(1-TE.success)).sum()/TE.w.sum(),3),'| sample: share failing',round(1-TE.success.mean(),3))
def reach(score,share,weights=None):
    w=np.ones(len(TE)) if weights is None else weights.values; o=np.argsort(score.values); cw=np.cumsum(w[o])/w.sum(); k=o[cw<=share]
    f=(1-TE.success.values)*w; return f[k].sum()/f.sum(), f[k].sum()/w[k].sum(), score.values[k].max()
rows=[]
for share in (.2,.3,.4,.5):
    a=reach(TE.high,share); b=reach(TE.low,share); c=reach(TE.low,share,TE.w)
    rows.append((share,round(a[0],3),round(b[0],3),round(b[1],3),round(c[0],3),round(c[1],3),round(c[2],3)))
V=pd.DataFrame(rows,columns=['coach capacity','sample: failures reached, rank by high end','sample: failures reached, rank by low end','sample: share of the group that fails','official mix: failures reached','official mix: share of the group that fails','low-end chance at the cut']); print(V.T.to_string()); V.to_csv(out+'/verdict_capacity.csv',index=False)
# within-category check: ranking by low end keeps the within-category AUC
print('AUC within category: high',round(M.auc_within(TE.success,TE.high,TE.cat_id),4),'low',round(M.auc_within(TE.success,TE.low,TE.cat_id),4),'| within parent: high',round(M.auc_within(TE.success,TE.high,TE.parent_category),4),'low',round(M.auc_within(TE.success,TE.low,TE.parent_category),4),'| overall: high',round(roc_auc_score(TE.success,TE.high),4),'low',round(roc_auc_score(TE.success,TE.low),4))
# who is in the riskiest 30% under each ranking, by parent
TE['risk_hi']=TE.high.rank(pct=True)<=.3; TE['risk_lo']=TE.low.rank(pct=True)<=.3
print(TE.groupby('parent_category').agg(n=('success','size'),in_riskiest_by_high=('risk_hi','mean'),in_riskiest_by_low=('risk_lo','mean')).round(2).T.to_string())
cut30=reach(TE.low,.3,TE.w)[2]; ART['verdict']={'reference_cut':float(cut30),'adjust_gain':5.0,'default_capacity':0.30}; adv.cut=ART['verdict']

## 6) What to change
For each campaign the module tries one change at a time on the levers a founder controls (goal in steps down to the minimum budget, duration, launch day, video, pre-launch page, more preparation time, title and blurb length) and keeps the best option per lever. Here on 4,000 campaigns of 2025, with the minimum budget assumed at half the goal.

In [ ]:
# ---- 4) what to change: best single change for every 2025 campaign (minimum budget assumed = half the goal)
t0=time.time(); smp=TE.sample(4000,random_state=7); res=[]
for i,r in smp.iterrows():
    fx=adv.fixes(r); res.append((i,fx))
print('fixes s',round(time.time()-t0))
best=pd.DataFrame([{'i':i,'lever':fx[0]['lever'] if fx else 'none','gain':fx[0]['gain'] if fx else 0,'n':len(fx)} for i,fx in res]).set_index('i')
print('top fix by lever (share of campaigns):'); print(best.lever.value_counts(normalize=True).round(3).to_string())
print('gain of the best fix, points: median',best.gain.median(),'| p75',best.gain.quantile(.75),'| share >=5',round((best.gain>=5).mean(),3),'| >=10',round((best.gain>=10).mean(),3))
allfx=pd.DataFrame([dict(f,i=i) for i,fx in res for f in fx]); print('mean gain when the lever is offered:'); print(allfx.groupby('lever').gain.agg(['size','median','max']).round(1).to_string())
smp=smp.join(best); smp['risky']=smp.low<cut30
smp['verdict']=np.select([smp.risky,smp.gain>=5],['Coach first','Adjust'],'Launch')
print(smp.groupby('verdict').agg(n=('success','size'),share=('success',lambda s:len(s)/len(smp)),funded_in_sample=('success','mean'),low=('low','mean'),high=('high','mean')).round(3).to_string())
for g in (3,5,8,10): print('adjust threshold',g,'-> Adjust share among non-coach',round(((~smp.risky)&(smp.gain>=g)).sum()/(~smp.risky).sum(),3))
# direction check of the levers against the raw data (within category, 2023-24): does the model's advice agree with what we see?
print('monotonic check: share of campaigns where a 30% lower goal LOWERS the predicted chance:')
q=smp.copy(); q['log_goal_usd']-=np.log10(1/.7)*1; q['goal_ratio']+=np.log10(.7); q['log_goal_per_day']+=np.log10(.7); print(round((adv.chances(q)[1]<smp.high-1e-9).mean(),3))

**Interpretation:** for more than four in ten campaigns no single change adds 2 points. The largest associations are with a pre-launch page and more preparation time. Both stand for groundwork (building an audience before launch): opening a page the day before launch will not add these points.

## 7) Product types that do badly
The Market score mostly reflects what is being made. A very low score is a signal for the coach that the product type itself has rarely been funded on Kickstarter.

In [ ]:
# ---- 5) product types that do badly: Market area
low=TE[TE.score_Market<=15]; print('Market score <=15: n',len(low),'funded in sample',round(low.success.mean(),3),'| others',round(TE[TE.score_Market>15].success.mean(),3))
print(low.groupby('parent_category').size().sort_values(ascending=False).head(6).to_dict()); print(low.subcategory.value_counts().head(10).to_dict())

## 8) Explanation tree (class: decision tree)
A depth-3 tree on features a founder can read (no text score, no category codes). The app shows the campaign's path through it.

In [ ]:
# explanation tree on features a founder can read (no text score, no category codes)
EX=[f for f in FM if f not in ('text_score','cat_id','parent_c','country_c')]
t3=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=500,random_state=42).fit(TR[EX].fillna(-1),TR.success)
p=t3.predict_proba(TE[EX].fillna(-1))[:,1]; print('readable depth-3 tree: 2025 AUC within category',round(M.auc_within(TE.success,p,TE.cat_id),3),'overall',round(roc_auc_score(TE.success,p),3))
print(export_text(t3,feature_names=EX,decimals=2,show_weights=True))

## 9) Three example campaigns from 2025

In [ ]:
# ---- 7) three example campaigns
for i in [smp[smp.verdict=='Coach first'].index[3],smp[smp.verdict=='Adjust'].index[3],smp[smp.verdict=='Launch'].index[3]]:
    r=TE.loc[i]; print('\n',r['name'][:70],'|',r.parent_category,'/',r.subcategory,'| goal $',round(r.goal_usd),'| outcome:',['failed','funded'][r.success])
    print('  chance range',round(r.low*100),'-',round(r.high*100),'% | areas',{a:int(r['score_'+a]) for a in A.AREAS},'| verdict',smp.verdict.get(i))
    for f in adv.fixes(r): print('   +',f['gain'],'pts:',f['change'])
EX=[f for f in FM if f not in ('text_score','cat_id','parent_c','country_c')]
from sklearn.tree import DecisionTreeClassifier
ART['tree_readable']=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=500,random_state=42).fit(TR[EX].fillna(-1),TR.success); ART['tree_features']=EX

### Cohort check
Simulated cohorts of 40 campaigns drawn from 2025; the coach takes the 12 riskiest.

In [ ]:
# cohort-mode verdict mix: simulated cohorts of 40 campaigns drawn from 2025
rng=np.random.default_rng(1); r=[]
for _ in range(500):
    c=TE.sample(40,random_state=int(rng.integers(1e9))); k=c.low.rank(method='first')<=12; r.append((c[k].success.mean(),(1-c[k].success).sum()/max((1-c.success).sum(),1),c[~k].success.mean()))
r=np.array(r); print('cohorts of 40, coach takes 12 (30%): funded share among the 12:',r[:,0].mean().round(3),'| share of the cohort\'s failures among the 12:',r[:,1].mean().round(3),'(5th-95th pct',np.percentile(r[:,1],5).round(2),np.percentile(r[:,1],95).round(2),') | funded share among the other 28:',r[:,2].mean().round(3))

## 10) Save

In [ ]:
joblib.dump(ART,out+'/models_M.joblib'); TE[['id','low','high','success','parent_category','cat_id']+['score_'+a for a in A.AREAS]].to_parquet(out+'/reference_2025.parquet')

## Conclusions
1. Market and Founder explain more than half of the score; the levers a founder controls explain the rest.
2. Area scores separate campaigns well: weak areas go with clearly lower funded rates.
3. Verdict: Coach first = riskiest share of the cohort by rank (default 30%); Adjust = one change adds at least 5 points; Launch = otherwise.
4. In a cohort with Kickstarter's official mix, the riskiest 30% contain about half of all failures, and nearly all of them fail.
5. Advice is association, not cause.